# Prepare data - Brain Mapping Analysis

- retrives Allen Brain Atlas ontologies from semsql using OAK
- creates simplified obo format file

In [1]:
!uv pip install oaklib

Using Python 3.10.9 environment at: /Users/cjm/repos/boomer-py/.venv
Audited 1 package in 37ms


In [2]:
from oaklib import get_adapter
from oaklib.datamodels.vocabulary import IS_A, PART_OF

In [36]:
from boomer.model import KB, ProperSubClassOf, EquivalentTo

In [4]:
kb = KB()

In [5]:
!mkdir -p results

In [6]:
ONTS = ["emapa", "hba", "dhba", "mba", "dmba", "pba"]

In [7]:
RELS = [IS_A, PART_OF, "emapa#part:of"]
#RELS = None

In [63]:
BRAIN_TERMS = {
    "emapa": "EMAPA:16894",
    "uberon": "UBERON:0001016",   # nervous system
}

In [23]:
from collections import defaultdict

all_entities = set()
for ont in ONTS:
    with open(f"results/source-{ont}.obo", "w") as f:
        n = 0
        print(f"ontology: {ont}\n", file=f)
        adapter = get_adapter(f"sqlite:obo:{ont}")
        pmap = defaultdict(list)
        for s, _p, parent in adapter.relationships(predicates=RELS):
            if parent.lower().startswith(f"{ont}:"):
                pmap[s].append(parent)
        assert pmap, "no parents"
        if ont in BRAIN_TERMS:
            entities = list(adapter.descendants(BRAIN_TERMS[ont], RELS))
            if len(entities) < 10:
                raise ValueError(f"No descendants of {BRAIN_TERMS[ont]}")
        else:
            entities = list(adapter.entities())
        for t in entities:
            if not t.lower().startswith(f"{ont}:"):
                continue
            lbl = adapter.label(t)
            print("[Term]", file=f)
            print(f"id: {t}", file=f)
            print(f"name: {lbl or t}", file=f)
            all_entities.add(t)
            if lbl:
                kb.labels[t] = lbl
            for parent in pmap.get(t, []):
                if parent in entities:
                    n += 1
                    print(f"is_a: {parent}", file=f)
                    kb.facts.append(ProperSubClassOf(sub=t, sup=parent))
            for s in adapter.entity_aliases(t):
                print(f'synonym: "{s}" RELATED []', file=f)
            print("", file=f)
        print(f"Ontology: {ont}; parents={n}")

        

Ontology: emapa; parents=407
Ontology: hba; parents=1837
Ontology: dhba; parents=3316
Ontology: mba; parents=1326
Ontology: dmba; parents=2691
Ontology: pba; parents=258


In [10]:
import yaml
with open("results/kb.yaml", "w") as f:
    f.write(yaml.dump(kb.model_dump()))

In [19]:
uberon = adapter = get_adapter(f"sqlite:obo:uberon")

In [64]:
ubts = list(uberon.descendants(BRAIN_TERMS["uberon"], RELS))

In [65]:
len(ubts)

3241

In [66]:
u2x = defaultdict(set)

In [69]:
for m in uberon.sssom_mappings(ubts):
    assert m.predicate_id == "oio:hasDbXref"
    if m.subject_id.startswith("CL"):
        continue
    if m.subject_id.startswith("GO"):
        continue
    if m.subject_id.startswith("_"):
        continue
    if m.object_id.startswith("UBERON:"):
        print(m)
        assert False
    if not m.subject_id.startswith("UBERON:"):
        print(m.subject_id)
        assert False
    if m.object_id in all_entities:
        u2x[m.subject_id].add(m.object_id)
        

In [70]:
len(u2x)

1288

In [71]:
target_kb = KB(name="true mappings")

In [72]:
facts = []
for clique in u2x.values():
    clique = list(clique)
    for i in range(0, len(clique)):
        for j in range(i+1, len(clique)):
            facts.append(EquivalentTo(sub=clique[i], equivalent=clique[j]))

In [73]:
len(facts)

2688

In [74]:
facts[0:5]

[EquivalentTo(fact_type='EquivalentTo', sub='EMAPA:35998', equivalent='DHBA:10505'),
 EquivalentTo(fact_type='EquivalentTo', sub='EMAPA:35998', equivalent='HBA:4634'),
 EquivalentTo(fact_type='EquivalentTo', sub='DHBA:10505', equivalent='HBA:4634'),
 EquivalentTo(fact_type='EquivalentTo', sub='EMAPA:35352', equivalent='HBA:9249'),
 EquivalentTo(fact_type='EquivalentTo', sub='EMAPA:35352', equivalent='DMBA:17767')]

In [75]:
target_kb.facts = facts

In [76]:
import yaml
with open("results/target_kb.yaml", "w") as f:
    f.write(yaml.dump(target_kb.model_dump()))

In [77]:
type(kb).__name__

'KB'